# 040-optimised — Organise records to download / convert, second ("optimised") selection

Identical to nb `040` except that it reads and writes **the optimised selection folder**
`cfg["proc_data"]["gm_selection_optimised"]` (`data_processed/06_gm_selection/AvgSA_03_optimised/`)
instead of `cfg["proc_data"]["gm_selection"]`. The original selection's lists are never written.

**Provenance guard** — `cache_utils.verify(...)` on each upstream CSV from `033-…-optimised` against
the optimised `AvgSA_03_final_ensembles.pickle`. A `StaleCacheError` means re-run `033-…-optimised`.

**Upstream** (from `033-…-optimised`, in the optimised folder; each with a `.manifest.json`):
- `AvgSA_03_summary_esm_records_selected.csv`, `AvgSA_03_summary_ngasub_records_selected.csv`,
  `AvgSA_03_records_to_convert.csv`.

Also reads the already-downloaded state from `cfg["raw_data"]["esm_hdf5_folder"]` and
`cfg["raw_data"]["ngasub_folder"]` (shared with the original; records are physical files, so a
record already downloaded for the original selection counts as available here too).

**Output** (in the optimised folder; each stamped with a `.manifest.json`):
- `records_availability_summary.csv`, `complete_site_im_sets.csv`,
  `ngasub_rsns_to_download_by_site_completion_batched.csv` (from `summarise_availability`).
- `esm_records_to_convert.csv`, `ngasub_records_to_convert.csv` — the conversion lists that
  `042-…-optimised` reads.

**Run order** — run top to bottom after `033-…-optimised`.

In [ ]:
import csv
import pandas as pd

from phd_project.scripts.cache_utils import fingerprint, verify, write_manifest
from phd_project.scripts.data_handling.summarise_record_availability import summarise_availability
from phd_project.config import config

cfg = config.load_config()

## Provenance guard

Verify the upstream selection CSVs are fresh before building any lists.

In [ ]:
# Provenance guard: refuse to build download/convert lists from stale selection
# CSVs. Each CSV was stamped by nb 033-optimised against the OPTIMISED
# final_ensembles.pickle; verify that stamp still matches before consuming it.
# A StaleCacheError here means you must re-run nb 033-optimised.

gm_selection_dir = cfg["proc_data"]["gm_selection_optimised"]   # NOT the original folder
selection_campaigns = ["AvgSA_03"]

for campaign in selection_campaigns:
    src = fingerprint(final_ensembles=gm_selection_dir / f"{campaign}_final_ensembles.pickle")
    for name in (f"{campaign}_summary_esm_records_selected.csv",
                 f"{campaign}_summary_ngasub_records_selected.csv",
                 f"{campaign}_records_to_convert.csv"):
        verify(gm_selection_dir / name, src)
print("Upstream selection CSVs verified fresh.")

## Build per-group download lists + NGA-Sub batchfiles

For each group (`combined`, `AvgSA_06`, `AvgSA_03`): combine the campaigns' selected records, drop those already downloaded, write a per-group NGA-Sub batchfile, and (for `combined`) the combined ESM/NGA-Sub download lists.

In [ ]:
# # --- groups to process (overlap between AvgSA_03 and AvgSA_06 is intentional) ---
groups = {
    # "combined": ["AvgSA_06", "AvgSA_03"],
    # "AvgSA_06": ["AvgSA_06"],
    "AvgSA_03": ["AvgSA_03"],
}

gm_dir = cfg["proc_data"]["gm_selection_optimised"]   # NOT the original folder

# # # combined download-CSV paths (written for the "combined" group only, unchanged names)
# # esm_combined_fp = gm_dir / f"all_esm_records_to_download.csv"
# # ngasub_combined_fp = gm_dir / f"all_ngasub_records_to_download.csv"

# # --- already-downloaded sets (computed once) ---
# # ESM: filenames look like  {event_id}__{station_code}__{location_code}.h5
# downloaded_esm = set()
# for f in cfg["raw_data"]["esm_hdf5_folder"].glob("*.h5"):
#     parts = f.stem.split("__")
#     if len(parts) == 3:
#         ev, st, loc = parts
#         downloaded_esm.add((ev, st, int(loc)))  # normalise "00" -> 0

# # NGAsub: subdirs named NGASub_RSN_<rsn>
# downloaded_ngasub = {
#     int(d.name.removeprefix("NGASub_RSN_"))
#     for d in cfg["raw_data"]["ngasub_folder"].iterdir()
#     if d.name.startswith("NGASub_RSN_")
# }


# def process_group(label, campaigns, write_combined_csvs=False):
#     # --- ESM: combine the selected records from the group's campaigns ---
#     esm_selected = pd.concat(
#         [pd.read_csv(gm_dir / f"{c}_summary_esm_records_selected.csv", header=[0, 1])
#          for c in campaigns],
#         ignore_index=True,
#     ).drop_duplicates()

#     esm_keys = list(zip(
#         esm_selected[("metadata", "event_id")],
#         esm_selected[("metadata", "station_code")],
#         esm_selected[("metadata", "location_code")].astype(int),
#     ))
#     esm_to_download = esm_selected[[k not in downloaded_esm for k in esm_keys]].reset_index(drop=True)

#     # --- NGAsub: combine the selected records from the group's campaigns ---
#     nga_selected = pd.concat(
#         [pd.read_csv(gm_dir / f"{c}_summary_ngasub_records_selected.csv", header=0)
#          for c in campaigns]
#     )
#     nga_selected = (nga_selected.groupby("NGAsubRSN")["count"].sum()
#                     .reset_index().sort_values("count", ascending=False))
#     nga_to_download = nga_selected[~nga_selected["NGAsubRSN"].isin(downloaded_ngasub)].reset_index(drop=True)

#     # batched RSN file: 30 RSNs per row, first cell = 1-indexed row number, no header
#     ngasub_rsn_batches_fp = gm_dir / f"{label}_ngasub_rsns_to_download_batched.csv"
#     rsns = nga_to_download["NGAsubRSN"].tolist()
#     rsn_rows = [[i // 30 + 1, *rsns[i:i + 30]] for i in range(0, len(rsns), 30)]
#     with open(ngasub_rsn_batches_fp, "w", newline="") as fh:
#         csv.writer(fh).writerows(rsn_rows)
#     # provenance: this batchfile derives only from the group's ngasub summaries
#     write_manifest(ngasub_rsn_batches_fp, fingerprint(**{
#         f"{c}_ngasub_selected": gm_dir / f"{c}_summary_ngasub_records_selected.csv"
#         for c in campaigns}))

#     # # combined download CSVs (written for the "combined" group only, unchanged names)
#     # if write_combined_csvs:
#     #     esm_to_download.to_csv(esm_combined_fp, index=False)
#     #     write_manifest(esm_combined_fp, fingerprint(**{
#     #         f"{c}_esm_selected": gm_dir / f"{c}_summary_esm_records_selected.csv"
#     #         for c in campaigns}))
#     #     nga_to_download.to_csv(ngasub_combined_fp, index=False)
#     #     write_manifest(ngasub_combined_fp, fingerprint(**{
#     #         f"{c}_ngasub_selected": gm_dir / f"{c}_summary_ngasub_records_selected.csv"
#     #         for c in campaigns}))

#     # --- summary ---
#     print(f"=== {label} ({' + '.join(campaigns)}) ===")
#     print("Records still to download (selected minus already downloaded):")
#     print(f"  ESM:    {len(esm_to_download):>5} / {len(esm_selected)} selected  "
#           f"({len(esm_selected) - len(esm_to_download)} already downloaded)")
#     print(f"  NGAsub: {len(nga_to_download):>5} / {len(nga_selected)} selected  "
#           f"({len(nga_selected) - len(nga_to_download)} already downloaded)")
#     print(f"  NGAsub batched RSN file: {ngasub_rsn_batches_fp.name}  "
#           f"({len(rsn_rows)} rows of up to 30)")
#     print()


# for label, campaigns in groups.items():
#     process_group(label, campaigns, write_combined_csvs=(label == "combined"))

In [ ]:
# get a record availability summary for each site and prioritise the downloads.
# Both the input folder (where AvgSA_03_final_ensembles.pickle is looked up) and the
# output folder are the OPTIMISED selection folder.
gm_selection_dir = cfg["proc_data"]["gm_selection_optimised"]
ngasub_folder = cfg["raw_data"]["ngasub_folder"]
esm_folder = cfg["raw_data"]["esm_hdf5_folder"]
site_model = cfg["hazard_models"]["eshm20_wp1_site_model"]
out_dir = gm_selection_dir
_ = summarise_availability(gm_selection_dir, ngasub_folder, esm_folder, site_model, out_dir)

## Records to convert

Combine the per-campaign conversion lists into the per-database AT2/HDF5→JSON conversion lists, and report the conversion counts per group.

In [ ]:
# paths to per-campaign conversion lists:
convert_AvgSA03_fp = gm_selection_dir / f"AvgSA_03_records_to_convert.csv"

esm_convert_fp = gm_selection_dir / f"esm_records_to_convert.csv"
ngasub_convert_fp = gm_selection_dir / f"ngasub_records_to_convert.csv"

conv_03 = pd.read_csv(convert_AvgSA03_fp, dtype=str)
conv_combined = pd.concat([conv_03], ignore_index=True).drop_duplicates()

# The ESM list needs the network code (added by nb 033 on 2026-09-30): the converter uses it to
# pick the right NET.STA group when one HDF5 file holds the same station for two networks.
# Refuse an old list without it rather than silently writing a list the converter would have to
# guess from.
if "network_code" not in conv_combined.columns:
    raise KeyError(f"{convert_AvgSA03_fp.name} has no network_code column: re-run nb 033")

# provenance: the combined convert lists always derive from both campaigns' lists
convert_provenance = fingerprint(
    AvgSA_03_records_to_convert=convert_AvgSA03_fp,
    # AvgSA_06_records_to_convert=convert_AvgSA06_fp,
)

# write the combined per-database conversion lists. ESM keeps network_code; NGA-Sub records are
# identified by the RSN alone, so its list is unchanged.
columns_per_db = {
    "ESM": ["record_identifier", "component", "network_code"],
    "NGASub": ["record_identifier", "component"],
}
for db_name, fp in [("ESM", esm_convert_fp), ("NGASub", ngasub_convert_fp)]:
    sub = (conv_combined[conv_combined["database"] == db_name]
           [columns_per_db[db_name]]
           .drop_duplicates()
           .reset_index(drop=True))
    if db_name == "ESM":
        # one row per (record_identifier, component) is what keeps the JSON names unique
        # (nb 033 already checks this; repeated here because this is the list the converter reads)
        dup = sub.duplicated(["record_identifier", "component"], keep=False)
        if dup.any():
            raise ValueError(f"ESM records listed under more than one network:\n{sub[dup]}")
        if sub["network_code"].isna().any():
            raise ValueError("ESM records without a network code:\n"
                             f"{sub[sub['network_code'].isna()]}")
    sub.to_csv(fp, index=False)
    write_manifest(fp, convert_provenance)
    print(f"{fp.name}: {len(sub)} records")

# --- per-group conversion counts (reuses `groups` from cell two) ---
print()
print("Records to convert per group:")
for label, campaigns in groups.items():
    conv = pd.concat(
        [pd.read_csv(gm_dir / f"{c}_records_to_convert.csv", dtype=str) for c in campaigns],
        ignore_index=True,
    ).drop_duplicates()
    counts = {
        db: len(conv[conv["database"] == db][["record_identifier", "component"]].drop_duplicates())
        for db in ("ESM", "NGASub")
    }
    print(f"  {label:>8}: ESM {counts['ESM']:>5}   NGASub {counts['NGASub']:>5}")